# AI Resume Analyzer on Google Colab
Run the Streamlit app from your GitHub repo and open it through a temporary public link.

**Steps:** run cell 1, 2, 3 in order (click the play button on each). Cell 3 prints a link ending in `trycloudflare.com`. Open it in a new tab.
The link stops working when you stop the notebook or Colab disconnects. Do not share your own API key in this notebook.

In [ ]:
# 1. Get the code and install the libraries (about 1-2 minutes)
REPO = "https://github.com/Sanjananyalamvar/ai-resume-analyzer.git"
!rm -rf /content/app && git clone -q $REPO /content/app
%cd /content/app
!pip install -q -r requirements.txt
!apt-get -qq install -y tesseract-ocr > /dev/null   # only needed to read photo resumes (OCR)
print("Done")

In [ ]:
# 2. (Optional) turn on AI feedback.
# Colab left menu -> key icon (Secrets) -> Add new secret: name GROQ_API_KEY, value = your key -> switch on "Notebook access".
import os
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("AI key found. Tick 'Enable AI features' in the app sidebar.")
except Exception:
    print("No key added. The app still works without the AI features.")

In [ ]:
# 3. Start the app and print the public link
import re, subprocess, time, urllib.request

urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", "/content/cloudflared")
!chmod +x /content/cloudflared

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
                  "--browser.gatherUsageStats", "false"], stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT)
time.sleep(8)

tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if found:
        print("Open your app here:", found.group(0))
        break
print("Keep this cell running while you use the app. Stop it with the square button when finished.")
tunnel.wait()

### Run the tests instead (optional)
```
!python tests/smoke_test_app.py
!python -m unittest discover -s tests -v
```
If the app does not open, run `!tail -30 /content/streamlit.log` in a new cell to see the error.